In [36]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import re
import syllables
import joblib

In [11]:
enron_ft = pd.read_csv('../data/enron_model_ft.csv', low_memory=False)

In [13]:
enron_ft = enron_ft.drop(columns=['Unnamed: 0'])

In [14]:
enron_ft.head()

,EmailSend,EmailReply,SubjectSend,SubjectReply,From,To,DateSend,DateReply,Context,Response_Within_30_Mins,...,syllable_count,avg_word_length,stop_word_ratio,response_time,Response_Within_15_Mins,Response_Within_60_Mins,Response_Within_120_Mins,Response_Within_3_hours,has_question,exclamation_count
0,"Nikki, Thanks for the note and I hope your doi...",Hello hello! I was so worried that I had said ...,Portland,Re: Portland,bill.williams@enron.com,-nikole@excite.com,2001-06-18 22:44:00,2001-06-19 15:49:00,NaN,0,...,717,3.920078,0.547758,1025.0,0,0,0,0,1,1
1,Jim Lokay Sales Representative British Parts I...,HI,Call when you receive this (no rush),RE: Call when you receive this (no rush),548@britishparts.com,michelle.lokay@enron.com,2002-03-19 08:30:00,2002-03-19 08:34:00,NaN,1,...,24,5.538462,0.000000,4.0,1,1,1,1,0,0
2,Congratulations! Have a good sleep.,Just woke up...thnx for your note. I believe t...,Thanks and,Re: Thanks and,louise.kitchen@enron.com,8774754543@skytel.com,2002-01-15 12:30:00,2002-01-15 16:29:00,NaN,0,...,9,5.800000,0.400000,239.0,0,0,0,0,0,1
3,Test,Call back : Geir.Solberg@enron.com|Test|Test,Test,Re: Test,geir.solberg@enron.com,8776519147@skytel.com,2002-01-19 10:55:00,2002-01-19 10:56:00,NaN,1,...,1,4.000000,0.000000,1.0,1,1,1,1,0,0
4,We are dropping a lot of marketers. It would b...,As shankman would say 'working ya',Marketers,RE: Marketers,8777865122@skytel.com,louise.kitchen@enron.com,2002-01-21 15:30:00,2002-01-21 15:44:00,NaN,1,...,20,3.187500,0.500000,14.0,1,1,1,1,0,0


In [32]:
# Arrange data to train model
enron_emails_3hr = enron_ft[['EmailSend','Response_Within_3_hours']]

from sklearn.model_selection import train_test_split

# X uses 'enron_emails_3hr' to keep the original DataFrame intact for your function later
X = enron_emails_3hr[['EmailSend']]
y = enron_emails_3hr['Response_Within_3_hours']

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
data = X_train.copy()
df = pd.DataFrame(data)

# 2. Enhanced Feature Engineering Function
def extract_text_metadata(dataframe):
    df_features = dataframe.copy()
    
    # Existing features
    df_features['char_count'] = df_features['EmailSend'].str.len()
    df_features['has_question'] = df_features['EmailSend'].str.contains('\?').astype(int)
    df_features['exclamation_count'] = df_features['EmailSend'].str.count('!')
    
    # Split email text into raw words/tokens for calculations
    # This lowers the case and splits by spaces (basic tokenization)
    words_series = df_features['EmailSend'].str.lower().str.split()
    
    # NEW FEATURE 1: Token Count (Total words)
    df_features['token_count'] = words_series.apply(len)
    
    # NEW FEATURE 2: Average Word Length
    # Divides total characters of all words by the number of words (prevents division by zero)
    df_features['avg_word_length'] = words_series.apply(
        lambda words: np.mean([len(w) for w in words]) if len(words) > 0 else 0
    )
    
    # NEW FEATURE 3: Number of Stop Words
    # Counts how many tokens in the message exist inside scikit-learn's official stop word set
    df_features['stop_word_count'] = words_series.apply(
        lambda words: sum(1 for w in words if w in ENGLISH_STOP_WORDS)
    )

    def fast_email_syllables(text):
            if pd.isna(text):
                return 0
            words = re.findall(r'\b\w+\b', str(text).lower())
            return sum(syllables.estimate(w) for w in words)

    df_features['syllable_count'] = df_features['EmailSend'].apply(fast_email_syllables)
    
    return df_features

# Enrich the dataset
df_enriched = extract_text_metadata(df)

# 3. Define Preprocessor
# Group all numerical features together so they are scaled uniformly 
num_features = ['char_count', 'exclamation_count', 'token_count', 'avg_word_length', 'stop_word_count', 'syllable_count']

preprocessor = ColumnTransformer(
    transformers=[
        ('text_vec', CountVectorizer(stop_words='english', lowercase=True), 'EmailSend'),
        ('num_scale', StandardScaler(), num_features)
    ],
    remainder='passthrough' # Keeps 'has_question' binary flag unscaled
)

# 4. Build and Train Pipeline
pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000))
])

# Create feature list for model input
all_feature_cols = ['EmailSend', 'has_question'] + num_features
X_train = df_enriched[all_feature_cols]
#y_train = df_enriched['Response_Within_3_hours']

pipeline.fit(X_train, y_train)

# 5. Predict on a Single New Email
new_email_text = "yo talk to me."

# Format and transform the new incoming message
new_data = pd.DataFrame({'EmailSend': [new_email_text]})
new_data_enriched = extract_text_metadata(new_data)

# Predict likelihood
probabilities = pipeline.predict_proba(new_data_enriched[all_feature_cols])
likelihood_of_response = probabilities[0][1]

print(f"Email: \"{new_email_text}\"")
print(f"Likelihood of a response within an hour: {likelihood_of_response:.2%}")

Email: "yo talk to me."
Likelihood of a response within an hour: 26.33%


In [37]:
joblib.dump(pipeline, "enron_model.pkl")
joblib.dump(df_enriched, "enron_data.pkl")

['enron_data.pkl']